In [1]:
from google.colab import files
uploaded = files.upload()

Saving departments.csv to departments.csv
Saving employees.csv to employees.csv
Saving employees_nested.json to employees_nested.json
Saving sales.csv to sales.csv


In [2]:
!ls

departments.csv  employees.csv	employees_nested.json  sales.csv  sample_data


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PySparkLabProject").getOrCreate()

In [11]:
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)

departments_df = spark.read.csv("departments.csv", header=True, inferSchema=True)

sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

nested_df = spark.read.option("multiline","true").json("employees_nested.json")

In [12]:
employees_df.show()
departments_df.show()
sales_df.show()
nested_df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+

+----------+---------+
|department| location|
+----------+---------+
|        IT|Bangalore|
|        HR|   Mumbai|
|   Finance|    Delhi|
+----------+---------+

+-------+------+--------+------+
|sale_id|emp_id| product|amount|
+-------+------+--------+------+
|      1|     1|  Laptop| 75000|
|      2|     2|   Mouse|  1500|
|      3|     3|Keyboard|  3000|
|      4|     1| Monitor| 12000|
|      5|     4|  Laptop| 75000|
|      6|     3|   Mouse|  1000|
|      7|     5|Keyboard|  1500|
|

In [13]:
employees_df.printSchema()
departments_df.printSchema()
sales_df.printSchema()
nested_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- joining_date: date (nullable = true)

root
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- emp_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [16]:
employees_df.show(5)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+
only showing top 5 rows


In [17]:
employees_df.count()

7

In [18]:
employees_df.columns

['emp_id', 'name', 'department', 'salary', 'joining_date']

In [19]:
employees_df.show(3)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
+------+-----+----------+------+------------+
only showing top 3 rows


In [20]:
employees_df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
|Rahul| 70000|
|Sneha| 60000|
|Arjun| 75000|
|Priya| 80000|
|Karan| 50000|
|Meera| 72000|
| Amit| 58000|
+-----+------+



In [21]:
employees_df.withColumnRenamed("salary", "emp_salary").show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     2|Sneha|        HR|     60000|  2022-11-15|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     4|Priya|   Finance|     80000|  2022-12-20|
|     5|Karan|        IT|     50000|  2023-02-05|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     7| Amit|        HR|     58000|  2023-01-18|
+------+-----+----------+----------+------------+



In [22]:
employees_df.filter(employees_df.joining_date > "2023-01-01").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [23]:
from pyspark.sql.functions import col


In [24]:
employees_df.filter(col("salary") > 65000).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     6|Meera|      NULL| 72000|  2023-04-10|
+------+-----+----------+------+------------+



In [25]:
employees_df.filter(col("department") == "IT").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+



In [26]:
employees_df.filter(
    (col("department") == "IT") & (col("salary") > 70000)
).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     3|Arjun|        IT| 75000|  2023-03-01|
+------+-----+----------+------+------------+



In [27]:
employees_df.drop("joining_date").show()

+------+-----+----------+------+
|emp_id| name|department|salary|
+------+-----+----------+------+
|     1|Rahul|        IT| 70000|
|     2|Sneha|        HR| 60000|
|     3|Arjun|        IT| 75000|
|     4|Priya|   Finance| 80000|
|     5|Karan|        IT| 50000|
|     6|Meera|      NULL| 72000|
|     7| Amit|        HR| 58000|
+------+-----+----------+------+



In [28]:
employees_df.orderBy("salary").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     5|Karan|        IT| 50000|  2023-02-05|
|     7| Amit|        HR| 58000|  2023-01-18|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     1|Rahul|        IT| 70000|  2023-01-10|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
+------+-----+----------+------+------------+



In [29]:
employees_df.orderBy(col("salary").desc()).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     4|Priya|   Finance| 80000|  2022-12-20|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     7| Amit|        HR| 58000|  2023-01-18|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+



In [30]:
employees_df.orderBy(col("salary").desc()).show(3)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     4|Priya|   Finance| 80000|  2022-12-20|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     6|Meera|      NULL| 72000|  2023-04-10|
+------+-----+----------+------+------------+
only showing top 3 rows


In [31]:
from pyspark.sql.functions import sum, avg, max, min

employees_df.select(sum("salary")).show()
employees_df.select(avg("salary")).show()
employees_df.select(max("salary")).show()
employees_df.select(min("salary")).show()

+-----------+
|sum(salary)|
+-----------+
|     465000|
+-----------+

+-----------------+
|      avg(salary)|
+-----------------+
|66428.57142857143|
+-----------------+

+-----------+
|max(salary)|
+-----------+
|      80000|
+-----------+

+-----------+
|min(salary)|
+-----------+
|      50000|
+-----------+



In [32]:
employees_df.groupBy("department").count().show()

employees_df.groupBy("department").avg("salary").show()

employees_df.groupBy("department").sum("salary").show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|      NULL|    1|
|   Finance|    1|
|        IT|    3|
+----------+-----+

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|        HR|    59000.0|
|      NULL|    72000.0|
|   Finance|    80000.0|
|        IT|    65000.0|
+----------+-----------+

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        HR|     118000|
|      NULL|      72000|
|   Finance|      80000|
|        IT|     195000|
+----------+-----------+



In [33]:
employees_df.join(departments_df, "department").show()

employees_df.join(departments_df, "department", "left").show()

employees_df.join(departments_df, "department") \
    .select("name", "location") \
    .show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     1|Rahul| 70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun| 75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya| 80000|  2022-12-20|    Delhi|
|        IT|     5|Karan| 50000|  2023-02-05|Bangalore|
|        HR|     7| Amit| 58000|  2023-01-18|   Mumbai|
+----------+------+-----+------+------------+---------+

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     1|Rahul| 70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun| 75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya| 80000|  2022-12-20|    Delhi|
|        IT|     5|Karan| 50000|  2023-02-05|Ba

In [34]:
from pyspark.sql.functions import col, lit, upper, length, when, to_date, year, month, explode, sum, avg

In [35]:
employees_df.withColumn("bonus", col("salary") * 0.10).show()

+------+-----+----------+------+------------+------+
|emp_id| name|department|salary|joining_date| bonus|
+------+-----+----------+------+------------+------+
|     1|Rahul|        IT| 70000|  2023-01-10|7000.0|
|     2|Sneha|        HR| 60000|  2022-11-15|6000.0|
|     3|Arjun|        IT| 75000|  2023-03-01|7500.0|
|     4|Priya|   Finance| 80000|  2022-12-20|8000.0|
|     5|Karan|        IT| 50000|  2023-02-05|5000.0|
|     6|Meera|      NULL| 72000|  2023-04-10|7200.0|
|     7| Amit|        HR| 58000|  2023-01-18|5800.0|
+------+-----+----------+------+------------+------+



In [36]:
employees_df.withColumn("company", lit("BotCampus")).show()

+------+-----+----------+------+------------+---------+
|emp_id| name|department|salary|joining_date|  company|
+------+-----+----------+------+------------+---------+
|     1|Rahul|        IT| 70000|  2023-01-10|BotCampus|
|     2|Sneha|        HR| 60000|  2022-11-15|BotCampus|
|     3|Arjun|        IT| 75000|  2023-03-01|BotCampus|
|     4|Priya|   Finance| 80000|  2022-12-20|BotCampus|
|     5|Karan|        IT| 50000|  2023-02-05|BotCampus|
|     6|Meera|      NULL| 72000|  2023-04-10|BotCampus|
|     7| Amit|        HR| 58000|  2023-01-18|BotCampus|
+------+-----+----------+------+------------+---------+



In [37]:
employees_df.filter(col("department").isNull()).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     6|Meera|      NULL| 72000|  2023-04-10|
+------+-----+----------+------+------------+



In [38]:
employees_df.fillna({"department": "Unknown"}).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|   Unknown| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [39]:
employees_df.dropna().show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [40]:
employees_df.withColumn("name_upper", upper(col("name"))).show()

+------+-----+----------+------+------------+----------+
|emp_id| name|department|salary|joining_date|name_upper|
+------+-----+----------+------+------------+----------+
|     1|Rahul|        IT| 70000|  2023-01-10|     RAHUL|
|     2|Sneha|        HR| 60000|  2022-11-15|     SNEHA|
|     3|Arjun|        IT| 75000|  2023-03-01|     ARJUN|
|     4|Priya|   Finance| 80000|  2022-12-20|     PRIYA|
|     5|Karan|        IT| 50000|  2023-02-05|     KARAN|
|     6|Meera|      NULL| 72000|  2023-04-10|     MEERA|
|     7| Amit|        HR| 58000|  2023-01-18|      AMIT|
+------+-----+----------+------+------------+----------+



In [41]:
employees_df.withColumn("name_length", length(col("name"))).show()

+------+-----+----------+------+------------+-----------+
|emp_id| name|department|salary|joining_date|name_length|
+------+-----+----------+------+------------+-----------+
|     1|Rahul|        IT| 70000|  2023-01-10|          5|
|     2|Sneha|        HR| 60000|  2022-11-15|          5|
|     3|Arjun|        IT| 75000|  2023-03-01|          5|
|     4|Priya|   Finance| 80000|  2022-12-20|          5|
|     5|Karan|        IT| 50000|  2023-02-05|          5|
|     6|Meera|      NULL| 72000|  2023-04-10|          5|
|     7| Amit|        HR| 58000|  2023-01-18|          4|
+------+-----+----------+------+------------+-----------+



In [42]:
employees_df.withColumn("salary_in_lakhs", col("salary") / 100000).show()

+------+-----+----------+------+------------+---------------+
|emp_id| name|department|salary|joining_date|salary_in_lakhs|
+------+-----+----------+------+------------+---------------+
|     1|Rahul|        IT| 70000|  2023-01-10|            0.7|
|     2|Sneha|        HR| 60000|  2022-11-15|            0.6|
|     3|Arjun|        IT| 75000|  2023-03-01|           0.75|
|     4|Priya|   Finance| 80000|  2022-12-20|            0.8|
|     5|Karan|        IT| 50000|  2023-02-05|            0.5|
|     6|Meera|      NULL| 72000|  2023-04-10|           0.72|
|     7| Amit|        HR| 58000|  2023-01-18|           0.58|
+------+-----+----------+------+------------+---------------+



In [43]:
employees_df.withColumn(
    "salary_grade",
    when(col("salary") >= 75000, "High")
    .when((col("salary") >= 60000) & (col("salary") <= 74999), "Medium")
    .otherwise("Low")
).show()

+------+-----+----------+------+------------+------------+
|emp_id| name|department|salary|joining_date|salary_grade|
+------+-----+----------+------+------------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|      Medium|
|     2|Sneha|        HR| 60000|  2022-11-15|      Medium|
|     3|Arjun|        IT| 75000|  2023-03-01|        High|
|     4|Priya|   Finance| 80000|  2022-12-20|        High|
|     5|Karan|        IT| 50000|  2023-02-05|         Low|
|     6|Meera|      NULL| 72000|  2023-04-10|      Medium|
|     7| Amit|        HR| 58000|  2023-01-18|         Low|
+------+-----+----------+------+------------+------------+



In [44]:
employees_date_df = employees_df.withColumn("joining_date", to_date(col("joining_date"), "yyyy-MM-dd"))
employees_date_df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [45]:
employees_date_df.withColumn("joining_year", year(col("joining_date"))) \
                 .withColumn("joining_month", month(col("joining_date"))) \
                 .show()

+------+-----+----------+------+------------+------------+-------------+
|emp_id| name|department|salary|joining_date|joining_year|joining_month|
+------+-----+----------+------+------------+------------+-------------+
|     1|Rahul|        IT| 70000|  2023-01-10|        2023|            1|
|     2|Sneha|        HR| 60000|  2022-11-15|        2022|           11|
|     3|Arjun|        IT| 75000|  2023-03-01|        2023|            3|
|     4|Priya|   Finance| 80000|  2022-12-20|        2022|           12|
|     5|Karan|        IT| 50000|  2023-02-05|        2023|            2|
|     6|Meera|      NULL| 72000|  2023-04-10|        2023|            4|
|     7| Amit|        HR| 58000|  2023-01-18|        2023|            1|
+------+-----+----------+------+------------+------------+-------------+



In [46]:
new_data = [
    (8, "Divya", "Finance", 67000, "2023-05-01"),
    (9, "Vikram", "IT", 73000, "2023-06-15")
]

new_columns = ["emp_id", "name", "department", "salary", "joining_date"]

new_employees_df = spark.createDataFrame(new_data, new_columns)
new_employees_df.show()

+------+------+----------+------+------------+
|emp_id|  name|department|salary|joining_date|
+------+------+----------+------+------------+
|     8| Divya|   Finance| 67000|  2023-05-01|
|     9|Vikram|        IT| 73000|  2023-06-15|
+------+------+----------+------+------------+



In [47]:
employees_df.union(new_employees_df).show()

+------+------+----------+------+------------+
|emp_id|  name|department|salary|joining_date|
+------+------+----------+------+------------+
|     1| Rahul|        IT| 70000|  2023-01-10|
|     2| Sneha|        HR| 60000|  2022-11-15|
|     3| Arjun|        IT| 75000|  2023-03-01|
|     4| Priya|   Finance| 80000|  2022-12-20|
|     5| Karan|        IT| 50000|  2023-02-05|
|     6| Meera|      NULL| 72000|  2023-04-10|
|     7|  Amit|        HR| 58000|  2023-01-18|
|     8| Divya|   Finance| 67000|  2023-05-01|
|     9|Vikram|        IT| 73000|  2023-06-15|
+------+------+----------+------+------------+



In [48]:
employees_df.unionByName(new_employees_df).show()

+------+------+----------+------+------------+
|emp_id|  name|department|salary|joining_date|
+------+------+----------+------+------------+
|     1| Rahul|        IT| 70000|  2023-01-10|
|     2| Sneha|        HR| 60000|  2022-11-15|
|     3| Arjun|        IT| 75000|  2023-03-01|
|     4| Priya|   Finance| 80000|  2022-12-20|
|     5| Karan|        IT| 50000|  2023-02-05|
|     6| Meera|      NULL| 72000|  2023-04-10|
|     7|  Amit|        HR| 58000|  2023-01-18|
|     8| Divya|   Finance| 67000|  2023-05-01|
|     9|Vikram|        IT| 73000|  2023-06-15|
+------+------+----------+------+------------+



In [49]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number

In [50]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

employees_df.withColumn("rank", rank().over(window_spec)).show()

+------+-----+----------+------+------------+----+
|emp_id| name|department|salary|joining_date|rank|
+------+-----+----------+------+------------+----+
|     6|Meera|      NULL| 72000|  2023-04-10|   1|
|     4|Priya|   Finance| 80000|  2022-12-20|   1|
|     2|Sneha|        HR| 60000|  2022-11-15|   1|
|     7| Amit|        HR| 58000|  2023-01-18|   2|
|     3|Arjun|        IT| 75000|  2023-03-01|   1|
|     1|Rahul|        IT| 70000|  2023-01-10|   2|
|     5|Karan|        IT| 50000|  2023-02-05|   3|
+------+-----+----------+------+------------+----+



In [51]:
employees_df.withColumn("row_number", row_number().over(window_spec)).show()

+------+-----+----------+------+------------+----------+
|emp_id| name|department|salary|joining_date|row_number|
+------+-----+----------+------+------------+----------+
|     6|Meera|      NULL| 72000|  2023-04-10|         1|
|     4|Priya|   Finance| 80000|  2022-12-20|         1|
|     2|Sneha|        HR| 60000|  2022-11-15|         1|
|     7| Amit|        HR| 58000|  2023-01-18|         2|
|     3|Arjun|        IT| 75000|  2023-03-01|         1|
|     1|Rahul|        IT| 70000|  2023-01-10|         2|
|     5|Karan|        IT| 50000|  2023-02-05|         3|
+------+-----+----------+------+------------+----------+



In [52]:
nested_df = spark.read.option("multiline", "true").json("employees_nested.json")
nested_df.show(truncate=False)
nested_df.printSchema()

+----------------------+------+-----+------------------------+
|address               |emp_id|name |skills                  |
+----------------------+------+-----+------------------------+
|{Hyderabad, Telangana}|1     |Rahul|[Python, SQL]           |
|{Bangalore, Karnataka}|2     |Sneha|[HR, Recruitment]       |
|{Chennai, Tamil Nadu} |3     |Arjun|[PySpark, Azure, Python]|
+----------------------+------+-----+------------------------+

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [53]:
nested_df.select(
    "emp_id",
    col("address.city").alias("city"),
    col("address.state").alias("state")
).show()

+------+---------+----------+
|emp_id|     city|     state|
+------+---------+----------+
|     1|Hyderabad| Telangana|
|     2|Bangalore| Karnataka|
|     3|  Chennai|Tamil Nadu|
+------+---------+----------+



In [54]:
nested_df.select("name", explode(col("skills")).alias("skill")).show()

+-----+-----------+
| name|      skill|
+-----+-----------+
|Rahul|     Python|
|Rahul|        SQL|
|Sneha|         HR|
|Sneha|Recruitment|
|Arjun|    PySpark|
|Arjun|      Azure|
|Arjun|     Python|
+-----+-----------+



In [55]:
sales_df.select(sum("amount").alias("total_sales")).show()

+-----------+
|total_sales|
+-----------+
|     244000|
+-----------+



In [56]:
sales_df.groupBy("emp_id").agg(sum("amount").alias("total_sales")).show()

+------+-----------+
|emp_id|total_sales|
+------+-----------+
|     1|     162000|
|     3|       4000|
|     5|       1500|
|     4|      75000|
|     2|       1500|
+------+-----------+



In [57]:
sales_df.groupBy("emp_id") \
       .agg(sum("amount").alias("total_sales")) \
       .orderBy(col("total_sales").desc()) \
       .show(1)

+------+-----------+
|emp_id|total_sales|
+------+-----------+
|     1|     162000|
+------+-----------+
only showing top 1 row


In [58]:
sales_df.groupBy("product").agg(sum("amount").alias("total_sales")).show()

+--------+-----------+
| product|total_sales|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       2500|
|Keyboard|       4500|
| Monitor|      12000|
+--------+-----------+



In [59]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [60]:
def categorize_sales(amount):
    if amount > 50000:
        return "Premium"
    elif 20000 <= amount <= 50000:
        return "Medium"
    else:
        return "Small"

sales_category_udf = udf(categorize_sales, StringType())

In [61]:
employees_pandas = employees_df.toPandas()
employees_pandas

,emp_id,name,department,salary,joining_date
0,1,Rahul,IT,70000,2023-01-10
1,2,Sneha,HR,60000,2022-11-15
2,3,Arjun,IT,75000,2023-03-01
3,4,Priya,Finance,80000,2022-12-20
4,5,Karan,IT,50000,2023-02-05
5,6,Meera,None,72000,2023-04-10
6,7,Amit,HR,58000,2023-01-18


In [62]:
employees_pandas[employees_pandas["salary"] > 65000]

,emp_id,name,department,salary,joining_date
0,1,Rahul,IT,70000,2023-01-10
2,3,Arjun,IT,75000,2023-03-01
3,4,Priya,Finance,80000,2022-12-20
5,6,Meera,None,72000,2023-04-10


In [63]:
import ipywidgets as widgets
from IPython.display import display

In [64]:
department_dropdown = widgets.Dropdown(
    options=["All", "IT", "HR", "Finance"],
    value="All",
    description="Department:"
)

display(department_dropdown)

Dropdown(description='Department:', options=('All', 'IT', 'HR', 'Finance'), value='All')

In [65]:
salary_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=100000,
    step=1000,
    description="Min Salary:"
)

display(salary_slider)

IntSlider(value=0, description='Min Salary:', max=100000, step=1000)

In [66]:
def filter_employees(department, min_salary):
    filtered_df = employees_df

    if department != "All":
        filtered_df = filtered_df.filter(col("department") == department)

    filtered_df = filtered_df.filter(col("salary") >= min_salary)
    filtered_df.show()

In [67]:
widgets.interactive(filter_employees, department=department_dropdown, min_salary=salary_slider)

interactive(children=(Dropdown(description='Department:', index=1, options=('All', 'IT', 'HR', 'Finance'), val…

In [71]:
#Final Mini Project

from pyspark.sql.functions import col, avg, sum

# Top Employee
top_employee = employees_df.orderBy(col("salary").desc()).first()["name"]

# Average Salary
average_salary = employees_df.select(avg("salary").alias("avg_salary")).first()["avg_salary"]

# Department Employee Count
department_counts = employees_df.groupBy("department").count().collect()

# Total Sales
total_sales = sales_df.select(sum("amount").alias("total_sales")).first()["total_sales"]

# Best Sales Employee
best_sales_row = sales_df.groupBy("emp_id") \
                         .agg(sum("amount").alias("total_sales")) \
                         .orderBy(col("total_sales").desc()) \
                         .first()

best_emp_id = best_sales_row["emp_id"]
best_sales_employee = employees_df.filter(col("emp_id") == best_emp_id).select("name").first()["name"]

# Create report text
report_text = f"""FINAL REPORT
Top Employee: {top_employee}
Average Salary: {average_salary}

Department Employee Count:
"""

for row in department_counts:
    report_text += f"{row['department']}: {row['count']}\n"

report_text += f"""
Total Sales: {total_sales}
Best Sales Employee: {best_sales_employee}
"""

print(report_text)

# Save to final_report.txt
with open("final_report.txt", "w") as file:
    file.write(report_text)

print("final_report.txt saved successfully.")



FINAL REPORT
Top Employee: Priya
Average Salary: 66428.57142857143

Department Employee Count:
HR: 2
None: 1
Finance: 1
IT: 3

Total Sales: 244000
Best Sales Employee: Rahul

final_report.txt saved successfully.
